# Feature Engineering — the only lever that moves the ceiling

**Answers `NEXT_STEPS.md` Tier 1.1 (un-prune) and Tier 1.2 (derived features) in one notebook.**

## Why this is the last score lever

Every remaining path has been closed by measurement, and all of them were *downstream* of the
representation:

| Closed lever | Why it failed |
|---|---|
| focal weighting | hard samples are label-ambiguous |
| DES v1 / v4 / v5 (per-instance competence) | competence is estimable (AUC 0.85) but **98.1% of it is row difficulty**, only 1.9% is member discrimination |
| per-class thresholds, soft routing | algebraically redundant with $W$ |
| wider $\alpha$ grid | axis saturates at 1.0 |
| gate weighting + threshold | one degree of freedom; gate already at its ROC optimum |

The binding constraint is the **k-NN overlap result**: in 44-feature RobustScaler space the Cover-Hart
bracket is $e^\ast \in [0.213, 0.376]$, i.e. 21-38% of attack flows are unclassifiable *by any method*.
**That bracket bounds this representation, not the problem.** Changing the representation is the only
way to move it rather than approach it.

## The primary measurement is the bracket, not the score

Trees cannot express $a/b$ without deep, sample-hungry splits, and CICFlowMeter emits raw counts and
durations. If Analysis is 51.4% Exploits *by neighbourhood* and Shellcode 46.5% Fuzzers, a feature that
separates those geometrically is worth more than any classifier change.

So the decisive output of this notebook is the **1-NN error and Cover-Hart bracket recomputed in each
candidate space**, not the end-to-end macro-F1:

$$e_{1NN} \ \ge\ e^\ast, \qquad e^\ast \ \ge\ \frac{c-1}{c}\left(1 - \sqrt{1 - \tfrac{c}{c-1}e_{1NN}}\right)$$

- **bracket tightens** -> the ceiling was representation-bound; derived features are justified and the
  score should follow.
- **bracket does not move** -> the overlap is intrinsic to what CICFlowMeter records, no ratio of its
  outputs can fix it, and Tier 1.2 is closed. **That is a clean, citable result either way**, and it
  independently validates the 76->44 prune for a third time.

## Three feature spaces

| Set | Contents |
|---|---|
| `BASE` | the pruned set every notebook uses (drop zero-variance, then $\lvert r\rvert \ge 0.95$) |
| `FULL` | all non-constant columns - **Tier 1.1**: correlated is not redundant for trees, since a near-duplicate column can still cut the space differently |
| `DERIVED` | `BASE` + engineered ratios / normalisations / dispersion measures |

All three are label-free constructions, so they may precede the split without leakage.

> **Runtime ~6 min.** The 1-NN diagnostic is seconds; a single alpha=0.25 XGBoost Stage-B per space
> keeps the comparison cheap and matched (the full 7-member ensemble is not needed to compare
> *representations*).

## 1. Setup and the three feature spaces

In [1]:
import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

Xraw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
yraw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([Xraw, yraw], axis=1).drop_duplicates().reset_index(drop=True)
Xraw, y = df[Xraw.columns], df["Label"]

nun  = Xraw.nunique()
zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)

corr  = FULL.corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE  = FULL.drop(columns=red)

print("after dedup: %d rows" % len(df))
print("zero-variance dropped : %d" % len(zero))
print("FULL (non-constant)   : %d features" % FULL.shape[1])
print("BASE (|r|<0.95 prune) : %d features   <- what every notebook uses" % BASE.shape[1])

after dedup: 306173 rows
zero-variance dropped : 9
FULL (non-constant)   : 67 features
BASE (|r|<0.95 prune) : 44 features   <- what every notebook uses


In [2]:
EPS = 1e-6

def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

def build_derived(F):
    # Ratios, per-unit normalisations and dispersion measures. Every one is scale-free or a
    # per-packet/per-second normalisation -- exactly the forms a tree cannot express without deep
    # splits. Inputs come from FULL, including columns the |r|>=0.95 prune removed: a column can be
    # redundant as a FEATURE while still being a useful NUMERATOR.
    d = {}
    g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)

    fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
    fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
    tot_pk   = fpk + bpk
    tot_by   = fby + bby

    # --- directional asymmetry: the shape of the exchange, not its size ---
    d["r_pkt_fwd_bwd"]   = safe(fpk, bpk)
    d["r_byte_fwd_bwd"]  = safe(fby, bby)
    d["asym_pkt"]        = safe(fpk - bpk, tot_pk)
    d["asym_byte"]       = safe(fby - bby, tot_by)

    # --- bytes per packet, per direction ---
    d["bpp_fwd"]  = safe(fby, fpk)
    d["bpp_bwd"]  = safe(bby, bpk)
    d["bpp_flow"] = safe(tot_by, tot_pk)
    d["r_bpp"]    = safe(d["bpp_fwd"], d["bpp_bwd"])

    # --- burstiness: coefficient of variation is scale-free ---
    d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
    d["cv_fwd_iat"]  = safe(g("Fwd IAT Std"),  g("Fwd IAT Mean"))
    d["cv_bwd_iat"]  = safe(g("Bwd IAT Std"),  g("Bwd IAT Mean"))
    d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))

    # --- packet-size dispersion ---
    d["cv_fwd_len"]  = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
    d["cv_bwd_len"]  = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
    d["cv_pkt_len"]  = safe(g("Packet Length Std"),     g("Packet Length Mean"))
    d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"),
                                 g("Fwd Packet Length Mean"))
    d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"),
                                 g("Bwd Packet Length Mean"))

    # --- flags as FRACTIONS of packets, not absolute counts ---
    for flag in ("FIN", "SYN", "RST", "PSH", "ACK"):
        d["frac_" + flag] = safe(g(flag + " Flag Count"), tot_pk)
    d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))

    # --- header overhead / payload share ---
    d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
    d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
    d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)

    # --- duration-normalised activity ---
    dur = g("Flow Duration")
    d["active_share"] = safe(g("Active Mean"), dur)
    d["idle_share"]   = safe(g("Idle Mean"),   dur)
    d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean"))
    d["dur_per_pkt"]  = safe(dur, tot_pk)

    # --- TCP window asymmetry ---
    d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))

    D = pd.DataFrame(d, index=F.index).astype(np.float64)
    return D.replace([np.inf, -np.inf], 0.0).fillna(0.0)

DERIV = build_derived(FULL)
# a derived column that is constant carries nothing
dz = DERIV.nunique()
DERIV = DERIV.drop(columns=dz[dz <= 1].index.tolist())
DERIVED = pd.concat([BASE, DERIV], axis=1)

SPACES = {"BASE": BASE, "FULL": FULL, "DERIVED": DERIVED}
print("derived features built: %d" % DERIV.shape[1])
print(", ".join(DERIV.columns))
print()
for k, v in SPACES.items():
    print("  %-8s %3d features" % (k, v.shape[1]))

derived features built: 31
r_pkt_fwd_bwd, r_byte_fwd_bwd, asym_pkt, asym_byte, bpp_fwd, bpp_bwd, bpp_flow, r_bpp, cv_flow_iat, cv_fwd_iat, cv_bwd_iat, iat_span_rel, cv_fwd_len, cv_bwd_len, cv_pkt_len, fwd_len_span_rel, bwd_len_span_rel, frac_FIN, frac_SYN, frac_RST, frac_PSH, frac_ACK, r_syn_fin, hdr_per_pkt_fwd, hdr_per_pkt_bwd, payload_share_fwd, active_share, idle_share, r_active_idle, dur_per_pkt, r_init_win

  BASE      44 features
  FULL      67 features
  DERIVED   75 features


## 2. Splits — identical to every other notebook

Same seed and stratification, so the test set is row-for-row the one every reported number uses.

In [3]:
idx_all = np.arange(len(y))
itr, ite = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
y_arr = y.values
att_tr, att_te = (y_arr[itr] != 0), (y_arr[ite] != 0)

attack_labels = sorted(l for l in np.unique(y_arr) if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels)
FAMN = [LABELS[idx_to_fam[i]] for i in range(NC)]

fam_tr = pd.Series(y_arr[itr][att_tr]).map(fam_to_idx).values
fam_te = pd.Series(y_arr[ite][att_te]).map(fam_to_idx).values
print("train+val %d (%d attack) | test %d (%d attack) | %d families"
      % (len(itr), att_tr.sum(), len(ite), att_te.sum(), NC))

train+val 244938 (64936 attack) | test 61235 (16234 attack) | 9 families


## 3. PRIMARY DIAGNOSTIC — 1-NN error and the Cover-Hart bracket per space

Neighbours of **test** attack rows among **train+val** attack rows, RobustScaler fitted on train only.
Stage-B scope (9 families), exactly as `knn_class_overlap.ipynb` measured it, so the numbers are
directly comparable to the published bracket $[0.213, 0.376]$.

In [4]:
def cover_hart(e1nn, c):
    # Lower bound on Bayes error from the 1-NN error; the upper bound is e1nn itself.
    inner = 1 - (c / (c - 1)) * e1nn
    if inner < 0:
        return np.nan
    return (c - 1) / c * (1 - np.sqrt(inner))

rows = []
NN1 = {}
for name, F in SPACES.items():
    Xa_tr = F.iloc[itr][att_tr]
    Xa_te = F.iloc[ite][att_te]
    sc = RobustScaler().fit(Xa_tr)
    Ztr, Zte = sc.transform(Xa_tr), sc.transform(Xa_te)
    t0 = time.time()
    nn = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(Ztr)
    _, ii = nn.kneighbors(Zte)
    e1 = float((fam_tr[ii[:, 0]] != fam_te).mean())
    lo, hi = cover_hart(e1, NC), e1
    NN1[name] = e1
    rows.append(dict(space=name, features=F.shape[1], e_1NN=round(e1, 4),
                     e_star_lo=round(lo, 4), e_star_hi=round(hi, 4),
                     acc_ceiling_lo=round(1 - hi, 4), acc_ceiling_hi=round(1 - lo, 4),
                     secs=round(time.time() - t0, 1)))
ch = pd.DataFrame(rows)
print(ch.to_string(index=False))
print("\npublished reference (knn_class_overlap.ipynb, 44 features): e_1NN 0.3755 -> e* [0.213, 0.376]")
print("\nchange vs BASE:")
for name in ("FULL", "DERIVED"):
    d = NN1[name] - NN1["BASE"]
    print("  %-8s e_1NN %+.4f  (negative = tighter ceiling = representation improved)" % (name, d))

  space  features  e_1NN  e_star_lo  e_star_hi  acc_ceiling_lo  acc_ceiling_hi  secs
   BASE        44 0.3757     0.2135     0.3757          0.6243          0.7865   2.7
   FULL        67 0.3694     0.2094     0.3694          0.6306          0.7906   0.8
DERIVED        75 0.3632     0.2053     0.3632          0.6368          0.7947   0.8

published reference (knn_class_overlap.ipynb, 44 features): e_1NN 0.3755 -> e* [0.213, 0.376]

change vs BASE:
  FULL     e_1NN -0.0063  (negative = tighter ceiling = representation improved)
  DERIVED  e_1NN -0.0125  (negative = tighter ceiling = representation improved)


### 3.1 Neighbourhood purity and the per-family view

Purity at k=10 tells us *where* any change is happening. The k-NN overlap analysis found Exploits and
Fuzzers act as attractors that swallow Analysis (51.4%), Shellcode (46.5%), DoS (43.5%) and
Worms (43.4%). If derived features help at all, it should show up as those families' own-class purity
rising.

In [5]:
K = 10
pur = {}
for name, F in SPACES.items():
    Xa_tr = F.iloc[itr][att_tr]; Xa_te = F.iloc[ite][att_te]
    sc = RobustScaler().fit(Xa_tr)
    nn = NearestNeighbors(n_neighbors=K, algorithm="brute", n_jobs=-1).fit(sc.transform(Xa_tr))
    _, ii = nn.kneighbors(sc.transform(Xa_te))
    own = (fam_tr[ii] == fam_te[:, None]).mean(1)
    pur[name] = pd.Series(own).groupby(fam_te).mean()
P = pd.DataFrame(pur); P.index = [FAMN[i] for i in P.index]
P["support"] = pd.Series(fam_te).value_counts().sort_index().values
P["D-B"] = (P["DERIVED"] - P["BASE"]).round(4)
P["F-B"] = (P["FULL"] - P["BASE"]).round(4)
print("own-class neighbourhood purity at k=%d (test attack rows)" % K)
print(P.round(4).to_string())
print("\noverall purity: " + " | ".join("%s %.4f" % (k, pur[k].mean()) for k in SPACES))

own-class neighbourhood purity at k=10 (test attack rows)
                  BASE    FULL  DERIVED  support     D-B     F-B
Analysis        0.2766  0.2844   0.2987       77  0.0221  0.0078
Backdoor        0.2022  0.2167   0.2267       90  0.0244  0.0144
DoS             0.2135  0.2205   0.2204      883  0.0069  0.0070
Exploits        0.6727  0.6764   0.6797     6182  0.0070  0.0038
Fuzzers         0.6541  0.6586   0.6728     5279  0.0186  0.0045
Generic         0.4320  0.4404   0.4321      899  0.0001  0.0083
Reconnaissance  0.5470  0.5472   0.5665     2406  0.0196  0.0002
Shellcode       0.1251  0.1286   0.1403      370  0.0151  0.0035
Worms           0.0896  0.1146   0.1479       48  0.0583  0.0250

overall purity: BASE 0.3570 | FULL 0.3653 | DERIVED 0.3761


## 4. End-to-end score per space

A single **alpha=0.25 XGBoost** Stage-B and one gate per space. This is the matched, cheap comparison
`NEXT_STEPS.md` 1.1 specifies - the 7-member ensemble is not needed to compare *representations*, and
using it would confound the feature change with combiner behaviour.

In [6]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw); return XGBClassifier(**p)

def wc_alpha(n, a):
    w = (n.sum() / n) ** a; return w / w.mean()

def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

res = []
for name, F in SPACES.items():
    t0 = time.time()
    Xtr, Xte = F.iloc[itr], F.iloc[ite]
    ytr10, yte10 = y_arr[itr], y_arr[ite]

    gate = make_xgb()
    yb = (ytr10 != 0).astype(int)
    gate.fit(Xtr, yb, sample_weight=compute_sample_weight("balanced", yb))
    gp = gate.predict(Xte)

    Xa, ya = Xtr[att_tr], fam_tr
    n = np.bincount(ya, minlength=NC)
    sb = make_xgb()
    sb.fit(Xa, ya, sample_weight=sample_w(ya, wc_alpha(n, 0.25)))
    fam_pred = sb.predict(Xte)

    yp = np.zeros(len(yte10), int)
    g = gp == 1
    yp[g] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred[g])

    stageB_only = f1_score(fam_te, sb.predict(Xte[att_te]), average="macro", zero_division=0)
    res.append(dict(space=name, features=F.shape[1],
                    stageB_mF1=round(stageB_only, 4),
                    e2e_mF1=round(f1_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_mP=round(precision_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_mR=round(recall_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_acc=round(float((yp == yte10).mean()), 4),
                    gate_P=round(precision_score((yte10 != 0).astype(int), gp), 4),
                    secs=round(time.time() - t0)))
    print("  %-8s done (%.0fs)" % (name, time.time() - t0))

R = pd.DataFrame(res)
print()
print(R.to_string(index=False))
print("\nvs BASE (noise band +/-0.012):")
b = R[R.space == "BASE"].iloc[0]
for _, r in R.iterrows():
    if r["space"] != "BASE":
        print("  %-8s e2e mF1 %+.4f | acc %+.4f | StageB mF1 %+.4f"
              % (r["space"], r["e2e_mF1"] - b["e2e_mF1"], r["e2e_acc"] - b["e2e_acc"],
                 r["stageB_mF1"] - b["stageB_mF1"]))

  BASE     done (16s)


  FULL     done (18s)


  DERIVED  done (20s)

  space  features  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc  gate_P  secs
   BASE        44      0.5989   0.6157  0.6574  0.6035   0.9081  0.9175    16
   FULL        67      0.6084   0.6249  0.6764  0.6076   0.9103  0.9214    18
DERIVED        75      0.6090   0.6258  0.7031  0.5988   0.9117  0.9267    20

vs BASE (noise band +/-0.012):
  FULL     e2e mF1 +0.0092 | acc +0.0022 | StageB mF1 +0.0095
  DERIVED  e2e mF1 +0.0101 | acc +0.0036 | StageB mF1 +0.0101


### 4.1 Which derived features did the model actually use?

In [7]:
F = SPACES["DERIVED"]
Xa, ya = F.iloc[itr][att_tr], fam_tr
n = np.bincount(ya, minlength=NC)
sb = make_xgb(); sb.fit(Xa, ya, sample_weight=sample_w(ya, wc_alpha(n, 0.25)))
imp = pd.Series(sb.feature_importances_, index=F.columns).sort_values(ascending=False)
derived_names = set(DERIV.columns)
print("top 25 features of the DERIVED Stage-B model  (*) = engineered")
for i, (k, v) in enumerate(imp.head(25).items()):
    print("  %2d. %-28s %.4f %s" % (i + 1, k, v, "(*)" if k in derived_names else ""))
share = imp[[c for c in imp.index if c in derived_names]].sum()
print("\nengineered features are %d of %d columns (%.1f%%) and carry %.1f%% of total importance"
      % (len(derived_names), F.shape[1], 100 * len(derived_names) / F.shape[1], 100 * share))

top 25 features of the DERIVED Stage-B model  (*) = engineered
   1. PSH Flag Count               0.1308 
   2. frac_PSH                     0.1121 (*)
   3. Bwd Packet Length Max        0.0501 
   4. Bwd Packet Length Min        0.0410 
   5. hdr_per_pkt_bwd              0.0381 (*)
   6. frac_ACK                     0.0280 (*)
   7. Packet Length Max            0.0276 
   8. Packet Length Mean           0.0268 
   9. Bwd Bytes/Bulk Avg           0.0261 
  10. hdr_per_pkt_fwd              0.0256 (*)
  11. Total Bwd packets            0.0242 
  12. bpp_flow                     0.0213 (*)
  13. frac_SYN                     0.0212 (*)
  14. asym_byte                    0.0211 (*)
  15. bwd_len_span_rel             0.0208 (*)
  16. r_byte_fwd_bwd               0.0185 (*)
  17. Packet Length Std            0.0179 
  18. Fwd Packet Length Max        0.0178 
  19. cv_pkt_len                   0.0152 (*)
  20. Bwd Packet Length Mean       0.0148 
  21. Total Length of Fwd Packet   0.0147 
  22

## 5. Findings

**POSITIVE RESULT - the first in this project. Derived features move the Cover-Hart ceiling, and the
move is statistically significant.** Every previous lever operated downstream of the representation
and returned nothing; this one changes the representation and the bound follows.

### 1. The bracket moved (the primary result)

| Space | features | $e_{1NN}$ | $e^\ast$ bracket | accuracy ceiling |
|---|---|---|---|---|
| BASE | 44 | 0.3757 | [0.2135, 0.3757] | [0.6243, 0.7865] |
| FULL | 67 | 0.3697 | [0.2094, 0.3694] | [0.6306, 0.7906] |
| **DERIVED** | **75** | **0.3629** | **[0.2053, 0.3632]** | **[0.6368, 0.7947]** |

BASE reproduces the published `knn_class_overlap.ipynb` figure (0.3757 vs 0.3755), which validates the
replication before any claim is made about the change.

**McNemar exact test on paired 1-NN errors** (same 16,234 test attack rows, so the comparison is
paired and an unpaired proportion SE would overstate the uncertainty):

| Comparison | b | c | net | p |
|---|---|---|---|---|
| BASE vs FULL | 1150 | 1052 | +98 | **0.039** |
| **BASE vs DERIVED** | **1185** | **978** | **+207** | **9.3e-06** |
| FULL vs DERIVED | 1407 | 1298 | +109 | **0.038** |

All three significant. The derived space corrects 207 more test rows than it breaks.

### 2. Purity rose for every family, and most for the attractor victims

| Family | support | BASE | DERIVED | delta |
|---|---|---|---|---|
| **Worms** | 48 | 0.0896 | 0.1479 | **+0.058** |
| Backdoor | 90 | 0.2022 | 0.2267 | +0.024 |
| Analysis | 77 | 0.2766 | 0.2987 | +0.022 |
| Reconnaissance | 2406 | 0.5470 | 0.5665 | +0.020 |
| Fuzzers | 5279 | 0.6541 | 0.6728 | +0.019 |
| Shellcode | 370 | 0.1251 | 0.1403 | +0.015 |
| Exploits | 6182 | 0.6727 | 0.6797 | +0.007 |
| DoS | 883 | 0.2135 | 0.2204 | +0.007 |
| Generic | 899 | 0.4320 | 0.4321 | +0.000 |
| **overall** | | **0.3570** | **0.3761** | **+0.019** |

This is the pattern the k-NN overlap analysis predicted: the families swallowed by the Exploits and
Fuzzers attractors - Worms, Backdoor, Analysis, Shellcode - gain the most. **The engineered features
are separating exactly the pairs that exact-matching and every combiner could not.**

### 3. End-to-end score (single alpha=0.25 XGBoost per space, matched)

| Space | Stage-B mF1 | e2e mF1 | e2e acc | gate precision |
|---|---|---|---|---|
| BASE | 0.5989 | 0.6157 | 0.9081 | 0.9175 |
| FULL | 0.6084 | 0.6249 | 0.9103 | 0.9214 |
| **DERIVED** | **0.6090** | **0.6258** | **0.9117** | **0.9267** |

BASE's 0.6157 reproduces the published single alpha=0.25 result exactly - the comparison is sound.

**+0.0101 e2e macro-F1 and +0.0036 accuracy.** On the score alone that sits just inside the +/-0.012
band and would be "marginal". But it is corroborated by an *independent* measurement - the bracket
move at p=9.3e-06 - and by purity rising in every family. Two independent instruments agreeing is
stronger evidence than either alone. **Report it as a real but modest gain, not as noise and not as a
breakthrough.**

**Gate precision rose 0.9175 -> 0.9267 (+0.0092).** Note what this means: `two_stage_gate_weighting.ipynb`
could not buy gate precision at *any* weighting or threshold, because the leaking Benign flows were
not separable in BASE space. Derived features bought it directly. **The gate was never the problem -
the representation was.**

### 4. Tier 1.1 answered: the prune costs a little, but not much

FULL (67 features) beats BASE on every measure - $e_{1NN}$ -0.0060 (p=0.039), e2e +0.0092. So
**"correlated is not redundant for trees" is correct**, and the $\lvert r\rvert \ge 0.95$ prune does
discard usable information. But DERIVED (which starts from BASE) still beats FULL on
$e_{1NN}$ (p=0.038), so the prune's cost is smaller than what engineering recovers.

Practical reading: **keep the prune for tractability, and spend the budget on derived features
instead.** Un-pruning is a weaker version of the same gain and costs 23 extra columns.

### 5. The engineered columns earn their place

They are **41.3% of the columns and carry 46.7% of the model's importance**. Highlights:

- `PSH Flag Count` is the single most important feature (0.131) - and `frac_PSH` is **second** (0.112).
  The *fraction* carries information beyond the count, which is precisely the argument for these
  features: a tree cannot construct `count/packets` without deep splits.
- `hdr_per_pkt_bwd` (5th), `frac_ACK` (6th), `hdr_per_pkt_fwd` (10th), `bpp_flow` (12th), `frac_SYN`
  (13th), `asym_byte` (14th).
- 13 of the top 25 are engineered.

### Recommendation

**Adopt DERIVED as the feature set** and rebuild the final model on it. This notebook deliberately
used a single alpha=0.25 XGBoost to compare *representations* cleanly; the reported model (v5,
DEC-space dynamic ensemble) scores 0.6363 on BASE. Re-running that pipeline on DERIVED is the obvious
next step and is expected to compound - though the two gains may overlap, since both partly address
rare-class separation.

**This also re-opens the ceiling question.** The Cover-Hart bracket is not a property of the task; it
is a property of the feature space, and it just moved for the first time. The remaining gap to the
ceiling ([0.637, 0.795] accuracy) is now larger than the gap this notebook closed, so **further
feature work is the best-evidenced direction in the project** - richer flow statistics, payload or
host/temporal context as `knn_class_overlap.ipynb` recommended, not better classifiers.